# **Bronze Layer - Origination Data Ingestion**

- **Project:** Mortgage Portfolio Product Governance & Performance Review
- **Layer:** Bronze (raw landing)
- **Source:** Freddie Mac Single-Family Loan-Level Dataset - 2018 origination sample (Pre-July 2026 layout)

This notebook lands the raw Freddie Mac origination file into the Lakehouse as a
Delta table ("bronze_origination"). It applies column names from the confirmed
file layout, adds lineage columns for governance, and reconciles the load before
recording the result in an audit table.

**What it does, step by step:**
1. Reads the pipe-delimited raw file, applying 32 column names in file order (all as text, to preserve source values faithfully)
2. Adds lineage columns - load batch, ingestion timestamp, source file
3. Writes the result to the "bronze_origination" Delta table
4. Reconciles raw line count against loaded rows to prove no data was lost during ingestion
5. Records the reconciliation outcome in a "load_audit" table

**Design principles:** Faithful bronze landing (type later in dbt) and auditable
loads (every load is lineage-tagged and reconciled).


### 1. Read and name columns

In [7]:
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql import Row

# applied column names to the raw file
# From schema/origination.py - 32 columns, exact file order (Pre-July 2026 layout)
origination_columns = [
    "credit_score", "first_payment_date", "first_time_homebuyer_flag",
    "maturity_date", "msa", "mi_pct", "number_of_units", "occupancy_status",
    "original_cltv", "original_dti", "original_upb", "original_ltv",
    "original_interest_rate", "channel", "ppm_flag", "amortization_type",
    "property_state", "property_type", "postal_code", "loan_sequence_number",
    "loan_purpose", "original_loan_term", "number_of_borrowers",
    "seller_name", "servicer_name", "super_conforming_flag",
    "pre_harp_loan_sequence_number", "program_indicator", "harp_indicator",
    "property_valuation_method", "interest_only_indicator", "mi_cancellation_indicator",
]

raw_path = "Files/raw/origination/sample_orig_2018.txt"

df = (spark.read
      .option("sep", "|")
      .option("header", "false")
      .option("inferSchema", "false")   # keep everything as strings - faithful bronze layer
      .csv(raw_path)
      .toDF(*origination_columns))       # apply names positionally

df.printSchema()

StatementMeta(, 03c0142e-8650-4856-960d-a761e2d9b884, 9, Finished, Available, Finished, False)

root
 |-- credit_score: string (nullable = true)
 |-- first_payment_date: string (nullable = true)
 |-- first_time_homebuyer_flag: string (nullable = true)
 |-- maturity_date: string (nullable = true)
 |-- msa: string (nullable = true)
 |-- mi_pct: string (nullable = true)
 |-- number_of_units: string (nullable = true)
 |-- occupancy_status: string (nullable = true)
 |-- original_cltv: string (nullable = true)
 |-- original_dti: string (nullable = true)
 |-- original_upb: string (nullable = true)
 |-- original_ltv: string (nullable = true)
 |-- original_interest_rate: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- ppm_flag: string (nullable = true)
 |-- amortization_type: string (nullable = true)
 |-- property_state: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- loan_sequence_number: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- original_loan_term: string (nullable 

### 2. Inspect and sanity-check

In [8]:
display(df.limit(10)) # first 10 rows
print("Row count:", df.count())

StatementMeta(, 03c0142e-8650-4856-960d-a761e2d9b884, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3117a91e-3ee9-4e4f-9359-6304ff125828)

Row count: 50000


## **Data Governance**

### 3. Add lineage columns

In [9]:
# added lineage columns for governance
df_bronze = (df
    .withColumn("load_batch_id", lit("2018_orig_001"))
    .withColumn("ingested_at_utc", current_timestamp())
    .withColumn("source_file", lit(raw_path)))

display(df_bronze.limit(3))

StatementMeta(, 03c0142e-8650-4856-960d-a761e2d9b884, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 698b7934-f555-4f7d-908b-e8412c581ad0)

### 4. Write to the bronze Delta table

In [10]:
# wrote df_broze as a delta table in the lakehouse
(df_bronze.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("bronze_origination"))
    

StatementMeta(, 03c0142e-8650-4856-960d-a761e2d9b884, 12, Finished, Available, Finished, False)

### 5. Reconcile the load

In [11]:
# reconcilation checks - to confirm raw data and loaded dataframe row count matches
raw_line_count = spark.read.text(raw_path).count()
loaded_count = spark.table("bronze_origination").count()

print("Raw lines:   ", raw_line_count)
print("Loaded rows: ", loaded_count)
print("Match:       ", raw_line_count == loaded_count)

StatementMeta(, 03c0142e-8650-4856-960d-a761e2d9b884, 13, Finished, Available, Finished, False)

Raw lines:    50000
Loaded rows:  50000
Match:        True


### 6. Record the result in the audit table

In [12]:
# wrote the reconciliation result to a load-audit table
audit = spark.createDataFrame([Row(
    load_batch_id   = "2018_orig_001",
    table_name      = "bronze_origination",
    source_file     = raw_path,
    raw_line_count  = raw_line_count,
    loaded_count    = loaded_count,
    reconciled      = bool(raw_line_count == loaded_count)
)])

(audit.write
    .mode("append")
    .format("delta")
    .saveAsTable("load_audit"))

display(spark.table("load_audit"))

StatementMeta(, 03c0142e-8650-4856-960d-a761e2d9b884, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ebb018d-9db8-4843-988b-95e7e34fdf48)